# Proyecto 5: Adquisiciones TI — Riesgo de Atraso en Proveedores
**Flujo:** Ingesta → Preparación → Split estratificado → S3 → Entrenamiento → Evaluación

**Requisito técnico:** `train_test_split` con `stratify`, integración S3 via `boto3`, reporte de Accuracy / Precision / Recall / Specificity.

## 1. Librerías

In [7]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    RocCurveDisplay,
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print(tf.config.list_physical_devices('GPU'))

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuración

In [8]:
HISTORY_PATH = "outputs/best_history.pkl"
CSV_PATH = "architecture/data/data.csv"
OUTPUT_DIR = "outputs"
MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.keras")
SCALER_PATH = os.path.join(OUTPUT_DIR, "scaler.pkl")
N_TRIALS = 5
TEST_SIZE = 0.20
RANDOM_STATE = 42

S3_BUCKET = "your-bucket-name"
S3_PREFIX = "proyecto5/splits/"
AWS_REGION = "us-east-1"

CATEGORICAL_COLS = ["category", "urgency", "region"]
DROP_COLS = ["contract_id"]
TARGET_COL = "delayed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Ingesta y Preparación
Carga del dataset con Pandas, inspección de estructura y distribución del target.

In [9]:
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print(f"Nulos: {df.isnull().sum().sum()}")
df.head()

Shape: (8000, 14)
Nulos: 0


,contract_id,category,budget_usd,compliance_history,urgency,contract_duration_days,num_line_items,lead_time_buffer_days,vendor_score,past_disputes,vendor_is_local,previous_contracts,region,delayed
0,CT-07670,Cloud License,48857.21,75,High,30,15,34,7.8,2,0,13,Asia,0
1,CT-03061,Services,12354.87,41,High,120,6,20,4.9,1,0,6,LATAM,1
2,CT-02151,Services,7719.48,64,Critical,180,3,16,5.6,1,1,4,LATAM,0
3,CT-03053,Services,54195.38,40,High,270,2,53,5.9,1,0,7,North America,0
4,CT-06454,Software,72390.69,85,High,45,12,15,8.5,1,0,9,Europe,0


In [10]:
print("Distribución del target:")
print(df[TARGET_COL].value_counts(normalize=True).rename({0: 'On time', 1: 'Delayed'}))

Distribución del target:
delayed
On time    0.513125
Delayed    0.486875
Name: proportion, dtype: float64


## 4. Preprocesamiento

In [11]:
def preprocess(df: pd.DataFrame):
    df = df.drop(columns=DROP_COLS)
    df = pd.get_dummies(df, columns=CATEGORICAL_COLS, drop_first=False, dtype=float)
    X = df.drop(columns=[TARGET_COL]).values.astype(np.float32)
    y = df[TARGET_COL].values.astype(np.float32)
    feature_names = df.drop(columns=[TARGET_COL]).columns.tolist()
    print(f"  Features after encoding : {X.shape[1]}")
    print(f"  Feature names           : {feature_names}\n")
    return X, y, feature_names

X, y, feature_names = preprocess(df)

  Features after encoding : 22
  Feature names           : ['budget_usd', 'compliance_history', 'contract_duration_days', 'num_line_items', 'lead_time_buffer_days', 'vendor_score', 'past_disputes', 'vendor_is_local', 'previous_contracts', 'category_Cloud License', 'category_Hardware', 'category_Networking', 'category_Services', 'category_Software', 'urgency_Critical', 'urgency_High', 'urgency_Low', 'urgency_Medium', 'region_Asia', 'region_Europe', 'region_LATAM', 'region_North America']



## 5. Split Estratificado
División `train / validation / test` preservando la proporción de clases con `stratify=y`.
El parámetro `stratify` garantiza que ambos splits mantienen la misma distribución del target.

In [12]:
def split_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    joblib.dump(scaler, SCALER_PATH)
    return X_train, X_test, y_train, y_test, scaler

X_train, X_test, y_train, y_test, scaler = split_data(X, y)
print(f"Train : {X_train.shape[0]} muestras | delayed={y_train.mean():.2%}")
print(f"Test  : {X_test.shape[0]} muestras  | delayed={y_test.mean():.2%}")

Train : 6400 muestras | delayed=48.69%
Test  : 1600 muestras  | delayed=48.69%


## 6. Subida a Amazon S3


In [13]:
def upload_splits_to_s3(X_train, X_test, y_train, y_test, feature_names):
    return

    # import boto3
    # s3 = boto3.client("s3", region_name=AWS_REGION)
    # splits = {
    #     "X_train.csv": pd.DataFrame(X_train, columns=feature_names),
    #     "X_test.csv":  pd.DataFrame(X_test,  columns=feature_names),
    #     "y_train.csv": pd.DataFrame(y_train, columns=["delayed"]),
    #     "y_test.csv":  pd.DataFrame(y_test,  columns=["delayed"]),
    # }
    # for filename, data in splits.items():
    #     local_path = os.path.join(OUTPUT_DIR, filename)
    #     data.to_csv(local_path, index=False)
    #     s3.upload_file(local_path, S3_BUCKET, S3_PREFIX + filename)
    #     print(f"  [S3] Uploaded {filename}")
    # s3.upload_file(SCALER_PATH, S3_BUCKET, S3_PREFIX + "scaler.pkl")

upload_splits_to_s3(X_train, X_test, y_train, y_test, feature_names)


## 7. Arquitectura de la Red Neuronal

In [14]:
def build_model(trial, n_features: int) -> keras.Model:
    n_layers      = trial.suggest_int("n_layers", 3, 4)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    model = keras.Sequential()
    model.add(layers.Input(shape=(n_features,)))

    for i in range(n_layers):
        units   = trial.suggest_categorical(f"units_l{i}", [16, 32, 64, 128])
        dropout = trial.suggest_float(f"dropout_l{i}", 0.4, 0.6)
        model.add(layers.Dense(units,
                               activation="relu",
                               kernel_regularizer=keras.regularizers.l2(
                                   trial.suggest_float("l2", 1e-5, 1e-2, log=True))))
        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))

    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 8. Optimización de Hiperparámetros con Optuna

In [20]:
def make_objective(X_train, y_train, n_features):
    def objective(trial):
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64,128])
        epochs = trial.suggest_int("epochs", 40, 100)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train,
            test_size=0.15,
            stratify=y_train,
            random_state=trial.number,
        )
        model = build_model(trial, n_features)
        early_stop = callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=0,
        )
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stop],
            verbose=0,
        )
        y_pred_prob = model.predict(X_val, verbose=0).ravel()
        return roc_auc_score(y_val, y_pred_prob)
    return objective

In [21]:
study = optuna.create_study(
    study_name="it_procurement_v3",
    storage="sqlite:///outputs/optuna_study.db",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    load_if_exists=True,
)
print(f"  Resuming from trial {len(study.trials)}" if len(study.trials) > 0 else "  Starting fresh")

study.optimize(
    make_objective(X_train, y_train, X_train.shape[1]),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

print(f"\nBest ROC-AUC: {study.best_value:.4f}")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-06-23 00:40:17,249] A new study created in RDB with name: it_procurement_v3


  Starting fresh


  0%|          | 0/5 [00:00<?, ?it/s]

[W 2026-06-23 00:40:17,357] Trial 0 failed with parameters: {'batch_size': 32, 'epochs': 49, 'n_layers': 3, 'learning_rate': 0.00013066739238053285, 'units_l0': 16, 'dropout_l0': 0.5939819704323989, 'l2': 0.00314288089084011} because of the following error: AttributeError("module 'keras.src.ops' has no attribute 'zeros'").
Traceback (most recent call last):
  File "/home/javie/ML/CertificacionProjectoFinal/MLFoundationsVenv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_14814/1969056549.py", line 11, in objective
    model = build_model(trial, n_features)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_14814/3027008555.py", line 11, in build_model
    model.add(layers.Dense(units,
  File "/home/javie/ML/CertificacionProjectoFinal/MLFoundationsVenv/lib/python3.12/site-packages/keras/src/models/sequential.py", line 122, in add
    self._maybe_rebu

AttributeError: module 'keras.src.ops' has no attribute 'zeros'

## 9. Entrenamiento del Modelo Final

In [22]:
def train_best_model(best_params, X_train, y_train, n_features):
    fixed_trial   = optuna.trial.FixedTrial(best_params)
    model         = build_model(fixed_trial, n_features)
    early_stop    = callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=0)

    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH, "rb") as f:
            previous_history = pickle.load(f)
        initial_epoch = len(previous_history["loss"])
        print(f"  Resuming final training from epoch {initial_epoch}")
        model = keras.models.load_model(MODEL_PATH)
    else:
        initial_epoch = 0

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        initial_epoch=initial_epoch,
        epochs=best_params["epochs"],
        batch_size=best_params["batch_size"],
        callbacks=[early_stop],
        verbose=0,
    )

    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH, "rb") as f:
            previous_history = pickle.load(f)
        for key in history.history:
            history.history[key] = previous_history[key] + history.history[key]

    model.save(MODEL_PATH)
    with open(HISTORY_PATH, "wb") as f:
        pickle.dump(history.history, f)

    return model, history

model, history = train_best_model(study.best_params, X_train, y_train, X_train.shape[1])

ValueError: Record does not exist.

## 10. Evaluación y Métricas de Rendimiento

In [ ]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def evaluate(model, X_test, y_test, threshold=0.5):
    y_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    acc       = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    spec      = specificity_score(y_test, y_pred)
    auc       = roc_auc_score(y_test, y_prob)
    print("\n  Full classification report:")
    print(classification_report(y_test, y_pred, target_names=["On time", "Delayed"]))
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {precision:.4f}")
    print(f"  Recall      : {recall:.4f}")
    print(f"  Specificity : {spec:.4f}")
    print(f"  ROC-AUC     : {auc:.4f}")
    return {
        "accuracy": acc, "precision": precision,
        "recall": recall, "specificity": spec,
        "roc_auc": auc, "y_prob": y_prob, "y_pred": y_pred,
    }

results = evaluate(model, X_test, y_test)

## 11. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"],         label="Train loss")
axes[0].plot(history.history["val_loss"],     label="Val loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history.history["accuracy"],     label="Train acc")
axes[1].plot(history.history["val_accuracy"], label="Val acc")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_history.png"), dpi=150)
plt.show()

In [ ]:
y_prob = model.predict(X_test, verbose=0).ravel()
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax, name="Neural Net")
ax.set_title("ROC Curve — IT Procurement Delay Prediction")
plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve.png"), dpi=150)
plt.show()

In [ ]:
cm = confusion_matrix(y_test, results["y_pred"])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im)
labels = ["On time", "Delayed"]
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()